In [1]:
from hrs_botany.spectral_cleaning import SpectralCleanerConfig, SpectralCleaner
from hrs_botany.utils import load_enmap_band_meta_txt

ModuleNotFoundError: No module named 'hrs_botany'

In [19]:
from pathlib import Path

import json
import numpy as np
import pandas as pd



In [20]:
band_df = load_enmap_band_meta_txt("./data/enmap_downloads/enmap_spectral_bands.txt")
print(band_df.head())
print('Number of bands: ', len(band_df))

   BAND #  CW (nm)  FWHM (nm)
0       1   423.25        6.5
1       2   429.75        6.5
2       3   436.25        6.5
3       4   442.75        6.5
4       5   449.25        6.5
Number of bands:  244


In [21]:
output_path = "./data/training_data/training_dataset.parquet"
df_built = pd.read_parquet(output_path)

In [2]:
# In your notebook or script
import numpy as np
import pandas as pd
from pathlib import Path

from hrs_botany.spectral_cleaning import (
    SpectralCleaner, SpectralCleanerConfig,
    ENMAP_L2A_KEEP_RANGES, passband_keep_mask
)

# --- inputs you already have ---
# band_df with columns: ['BAND #', 'CW (nm)', 'FWHM (nm)']
# df_built with columns: metadata + band_1..band_N (N may be < full sensor)

# 1) Build config from full band table (center wavelengths)
band_df_sorted = band_df.sort_values("BAND #").reset_index(drop=True)
wl_all = band_df_sorted["CW (nm)"].to_numpy(float).tolist()

cfg = SpectralCleanerConfig(
    wl_nm=wl_all,
    keep_ranges_nm=ENMAP_L2A_KEEP_RANGES,   # standard EnMAP L2A keep windows
)

cleaner = SpectralCleaner(cfg)

# 2) OPTIONAL (recommended): use passband-aware mask aligned to the actual columns present in df_built
#    This handles cases where df_built has a subset of bands (e.g., 224 of 244).
band_cols = [c for c in df_built.columns if isinstance(c, str) and c.startswith("band_")]
band_cols.sort(key=lambda c: int(c.split("_")[1]))
present_idxs = np.array([int(c.split("_")[1]) for c in band_cols], dtype=int)  # 1-based

centers_all = band_df_sorted["CW (nm)"].to_numpy(float)
fwhm_all    = band_df_sorted["FWHM (nm)"].to_numpy(float)

centers_sub = centers_all[present_idxs - 1]
fwhm_sub    = fwhm_all[present_idxs - 1]

# install a passband-aware keep mask (replaces default center-based mask)
cleaner.band_keep_mask = passband_keep_mask(centers_sub, fwhm_sub, ENMAP_L2A_KEEP_RANGES)

# 3) Mask columns (rows unchanged)
df_masked = cleaner.fit_transform(df_built)

print("Shapes:", df_built.shape, "→", df_masked.shape)
print("Kept bands:", len([c for c in df_masked.columns if c.startswith("band_")]))


ModuleNotFoundError: No module named 'hrs_botany'

In [22]:
# Create config from your band_df
cfg = SpectralCleanerConfig(
    wl_nm=band_df["CW (nm)"].tolist(),
    keep_ranges_nm=[(430, 2400)]  # example range
)

cleaner = SpectralCleaner(cfg)
df_masked = cleaner.fit_transform(df_built)

print(df_built.shape, "→", df_masked.shape)  # same rows, fewer columns


(1559, 274) → (1559, 272)


In [23]:
df_masked

,plot_id,scene_id,pixel_id,row,col,centroid,2683936,2685580,5284871,2685796,...,band_215,band_216,band_217,band_218,band_219,band_220,band_221,band_222,band_223,band_224
0,0112,ENMAP01-____L2A-DT0000001029_20220609T193006Z_...,1_2,1,2,POINT (722850 4274700),1.0,1.0,1.0,NaN,...,1088,1036,1080,1046,1053,983,1036,997,1007,936
1,0112,ENMAP01-____L2A-DT0000001029_20220609T193006Z_...,2_2,2,2,POINT (722850 4274670),1.0,0.0,1.0,NaN,...,723,663,688,687,698,627,698,647,649,559
2,0112,ENMAP01-____L2A-DT0000001029_20220609T193006Z_...,2_3,2,3,POINT (722880 4274670),1.0,0.0,1.0,NaN,...,885,828,858,844,838,770,831,806,803,717
3,0112,ENMAP01-____L2A-DT0000041496_20230911T193100Z_...,1_2,1,2,POINT (722850 4274700),1.0,1.0,1.0,NaN,...,512,461,489,493,490,507,443,429,463,401
4,0112,ENMAP01-____L2A-DT0000041496_20230911T193100Z_...,2_2,2,2,POINT (722850 4274670),1.0,0.0,1.0,NaN,...,354,338,321,320,329,308,307,293,328,271
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1554,f1,ENMAP01-____L2A-DT0000007897_20230215T193730Z_...,14_11,14,11,POINT (582630 4096620),NaN,NaN,NaN,1.0,...,242,215,235,202,228,200,199,158,146,122
1555,f1,ENMAP01-____L2A-DT0000007897_20230215T193730Z_...,14_12,14,12,POINT (582660 4096620),NaN,NaN,NaN,1.0,...,248,236,240,186,185,218,204,123,172,69
1556,f1,ENMAP01-____L2A-DT0000007897_20230215T193730Z_...,14_13,14,13,POINT (582690 4096620),NaN,NaN,NaN,1.0,...,230,231,214,197,240,187,186,153,177,71
1557,f1,ENMAP01-____L2A-DT0000007897_20230215T193730Z_...,15_11,15,11,POINT (582630 4096590),NaN,NaN,NaN,1.0,...,289,279,284,266,267,245,241,177,197,136


   BAND #  CW (nm)  FWHM (nm)
0       1   423.25        6.5
1       2   429.75        6.5
2       3   436.25        6.5
3       4   442.75        6.5
4       5   449.25        6.5
Number of bands:  244


In [10]:
import numpy as np
import pandas as pd
from pathlib import Path

# --- precise keep mask using passband edges (center +/- FWHM/2) ---
def passband_keep_mask(centers_nm: np.ndarray,
                       fwhm_nm: np.ndarray,
                       keep_ranges_nm=((430,1330),(1460,1780),(1960,2450))) -> np.ndarray:
    """
    Keep a band if its passband [center - FWHM/2, center + FWHM/2] overlaps ANY keep range.
    """
    lo = centers_nm - 0.5 * fwhm_nm
    hi = centers_nm + 0.5 * fwhm_nm
    mask = np.zeros_like(centers_nm, dtype=bool)
    for a, b in keep_ranges_nm:
        mask |= (hi >= a) & (lo <= b)  # interval overlap
    return mask

def build_config_from_band_df(band_df: pd.DataFrame,
                              keep_ranges_nm=((430,1330),(1460,1780),(1960,2450)),
                              **cfg_overrides):
    """
    band_df columns: 'BAND #', 'CW (nm)', 'FWHM (nm)'.
    Returns (cfg, centers_nm, fwhm_nm, keep_mask).
    """
    # Ensure sorted by band number (just in case)
    dfb = band_df.sort_values("BAND #").reset_index(drop=True)

    centers_nm = dfb["CW (nm)"].to_numpy(dtype=float)           # shape (n_bands,)
    fwhm_nm    = dfb["FWHM (nm)"].to_numpy(dtype=float)

    # default wavelength-keep mask is center-based; we'll compute a passband mask too
    keep_mask_passband = passband_keep_mask(centers_nm, fwhm_nm, keep_ranges_nm)

    # Build the config (uses center wavelengths for indexing/order)
    cfg_kwargs = dict(
        wl_nm=centers_nm,
        keep_ranges_nm=keep_ranges_nm,
        hampel_window=5,
        hampel_nsig=4.0,
        sg_window=7,
        sg_poly=2,
        scene_robust_scale=True,
        groupby_col="scene_id",
        min_coverage_keep=0.7,
        drop_all_band_nan_rows=True,
        qa_drop_cols=None,
        band_prefix="band_",
        stats_path=Path("artifacts/spectral_clean_stats.json"),
    )
    cfg_kwargs.update(cfg_overrides)
    cfg = SpectralCleanerConfig(**cfg_kwargs)

    return cfg, centers_nm, fwhm_nm, keep_mask_passband

# --- optional: quick audit plot of keep mask vs wavelength ---
def plot_band_keep(band_df: pd.DataFrame,
                   keep_ranges_nm=((430,1330),(1460,1780),(1960,2450))):
    import matplotlib.pyplot as plt

    dfb = band_df.sort_values("BAND #").reset_index(drop=True)
    c = dfb["CW (nm)"].to_numpy(float)
    w = dfb["FWHM (nm)"].to_numpy(float)
    keep = passband_keep_mask(c, w, keep_ranges_nm)

    fig, ax = plt.subplots(figsize=(10, 2.4))
    # draw passbands as thin rectangles; color by keep
    for ci, wi, k in zip(c, w, keep):
        ax.add_patch(plt.Rectangle((ci - wi/2, 0), wi, 1, alpha=0.8, ec="k", lw=0.2, fc=("C0" if k else "0.85")))
    # shade keep windows
    for a, b in keep_ranges_nm:
        ax.axvspan(a, b, alpha=0.08)
    ax.set_xlim(c.min()-20, c.max()+20)
    ax.set_ylim(0, 1)
    ax.set_yticks([])
    ax.set_xlabel("Wavelength (nm)")
    ax.set_title("Bands kept (colored) vs dropped (light gray)")
    plt.tight_layout()
    return fig


In [11]:
# Create config from your band_df
cfg = SpectralCleanerConfig(
    wl_nm=band_df["CW (nm)"].tolist(),
    keep_ranges_nm=[(430, 2400)]  # example range
)

cleaner = SpectralCleaner(cfg)
df_masked = cleaner.fit_transform(df_built)

print(df_built.shape, "→", df_masked.shape)  # same rows, fewer columns


NameError: name 'df_built' is not defined

In [12]:
# 1) Build config from your band table
cfg, wl_nm, fwhm_nm, keep_mask_passband = build_config_from_band_df(band_df)

# 2) (Optional) Visual audit
fig = plot_band_keep(band_df)

# 3) Initialize cleaner
cleaner = SpectralCleaner(cfg)



TypeError: SpectralCleanerConfig.__init__() got an unexpected keyword argument 'hampel_window'

In [8]:
def cfg_from_df_and_band_table(df_built, band_df, **overrides):
    """
    Align cfg.wl_nm and keep-ranges to the actual band indices present in df_built.
    band_df columns: ['BAND #','CW (nm)','FWHM (nm)'] for the full sensor.
    """
    # 1) parse actual band indices present in df_built
    band_cols = sorted([c for c in df_built.columns if c.startswith("band_")],
                       key=lambda s: int(s.split("_")[1]))
    idxs = np.array([int(c.split("_")[1]) for c in band_cols])  # 1-based

    # 2) pull centers/FWHM for exactly those indices
    band_df = band_df.sort_values("BAND #")
    centers_all = band_df["CW (nm)"].to_numpy(float)
    fwhm_all    = band_df["FWHM (nm)"].to_numpy(float)

    wl_nm  = centers_all[idxs - 1]
    fwhm_nm = fwhm_all[idxs - 1]

    # 3) build passband-aware keep mask over the subset
    def passband_keep_mask(centers_nm, fwhm_nm, keep_ranges_nm):
        lo = centers_nm - 0.5 * fwhm_nm
        hi = centers_nm + 0.5 * fwhm_nm
        mask = np.zeros_like(centers_nm, dtype=bool)
        for a, b in keep_ranges_nm:
            mask |= (hi >= a) & (lo <= b)
        return mask

    keep_ranges_nm = overrides.pop("keep_ranges_nm", ((430,1330),(1460,1780),(1960,2450)))
    passband_mask = passband_keep_mask(wl_nm, fwhm_nm, keep_ranges_nm)

    # 4) make config aligned to df_built
    cfg = SpectralCleanConfig(
        wl_nm=wl_nm,
        keep_ranges_nm=keep_ranges_nm,
        **overrides
    )

    # 5) init cleaner and inject the passband-aware mask (optional but nicer)
    cleaner = SpectralCleaner(cfg)
    cleaner.band_cols = band_cols                      # fix the order explicitly
    cleaner.band_keep_mask = passband_mask             # use passband-aware keep
    return cleaner, wl_nm, fwhm_nm, passband_mask


In [12]:

# Usage:
cleaner, wl_nm_sub, fwhm_nm_sub, keep_mask = cfg_from_df_and_band_table(
    df_built,
    band_df,
    sg_window=7, sg_poly=2,
    scene_robust_scale=True,
    groupby_col="scene_id",
    min_coverage_keep=0.0,
    stats_path=Path("artifacts/spectral_clean_stats.json"),
)
df_clean = cleaner.fit_transform(df_built)


/Users/dangause/Desktop/calacademy/hrs_botany/hrs-botany/src/hrs_botany/spectral_cleaning.py:71: RuntimeWarning: All-NaN slice encountered
  med = np.nanmedian(w)
/Users/dangause/Desktop/calacademy/hrs_botany/hrs-botany/src/hrs_botany/spectral_cleaning.py:72: RuntimeWarning: All-NaN slice encountered
  mad = np.nanmedian(np.abs(w - med))


▸ Dropped 1324 rows with all NaNs on kept bands


/Users/dangause/Desktop/calacademy/hrs_botany/hrs-botany/src/hrs_botany/spectral_cleaning.py:81: RuntimeWarning: All-NaN slice encountered
  med = np.nanmedian(X, axis=0)
/Users/dangause/Desktop/calacademy/hrs_botany/hrs-botany/.venv/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:1620: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,


In [13]:
df_built

,plot_id,scene_id,pixel_id,row,col,centroid,band_1,band_2,band_3,band_4,...,2876032,2878124,2880791,2882802,3024146,3176787,3189835,5285497,5371727,6362957
0,0112,ENMAP01-____L2A-DT0000001029_20220609T193006Z_...,1_2,1,2,POINT (722850 4274700),183,186,186,203,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0112,ENMAP01-____L2A-DT0000001029_20220609T193006Z_...,2_2,2,2,POINT (722850 4274670),142,134,135,167,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0112,ENMAP01-____L2A-DT0000001029_20220609T193006Z_...,2_3,2,3,POINT (722880 4274670),138,133,133,166,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0112,ENMAP01-____L2A-DT0000041496_20230911T193100Z_...,1_2,1,2,POINT (722850 4274700),130,102,86,127,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0112,ENMAP01-____L2A-DT0000041496_20230911T193100Z_...,2_2,2,2,POINT (722850 4274670),110,78,60,106,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1554,f1,ENMAP01-____L2A-DT0000007897_20230215T193730Z_...,14_11,14,11,POINT (582630 4096620),117,87,66,97,...,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1555,f1,ENMAP01-____L2A-DT0000007897_20230215T193730Z_...,14_12,14,12,POINT (582660 4096620),131,88,63,87,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1556,f1,ENMAP01-____L2A-DT0000007897_20230215T193730Z_...,14_13,14,13,POINT (582690 4096620),147,94,59,79,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1557,f1,ENMAP01-____L2A-DT0000007897_20230215T193730Z_...,15_11,15,11,POINT (582630 4096590),127,102,60,119,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


In [14]:
df_clean

,plot_id,scene_id,pixel_id,row,col,centroid,band_1,band_2,band_3,band_4,...,2876032,2878124,2880791,2882802,3024146,3176787,3189835,5285497,5371727,6362957
0,0069,ENMAP01-____L2A-DT0000028166_20230709T064108Z_...,2_7,2,7,POINT (235800 4230660),NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0069,ENMAP01-____L2A-DT0000028166_20230709T064108Z_...,2_8,2,8,POINT (235830 4230660),NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0069,ENMAP01-____L2A-DT0000028166_20230709T064108Z_...,3_7,3,7,POINT (235800 4230630),NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0069,ENMAP01-____L2A-DT0000028166_20230709T064108Z_...,3_8,3,8,POINT (235830 4230630),NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0069,ENMAP01-____L2A-DT0000028166_20230709T064108Z_...,3_9,3,9,POINT (235860 4230630),NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
230,f1,ENMAP01-____L2A-DT0000003707_20220921T192716Z_...,11_8,11,8,POINT (582540 4096710),NaN,NaN,NaN,NaN,...,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
231,f1,ENMAP01-____L2A-DT0000007897_20230215T193730Z_...,10_14,10,14,POINT (582720 4096740),NaN,NaN,NaN,NaN,...,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
232,f1,ENMAP01-____L2A-DT0000007897_20230215T193730Z_...,12_12,12,12,POINT (582660 4096680),NaN,NaN,NaN,NaN,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
233,f1,ENMAP01-____L2A-DT0000007897_20230215T193730Z_...,13_10,13,10,POINT (582600 4096650),NaN,NaN,NaN,NaN,...,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [15]:

# 4) Fit/transform your built dataset
df_clean = cleaner.fit_transform(df_built)   # df_built has band_1..band_244 and scene_id

# 5) Save stats for reuse on test data
cleaner.save_stats()

ValueError: Band count mismatch: found 224 band columns, but cfg.wl_nm has 244 wavelengths.

In [19]:
sum(df_built.columns.str.startswith("band_"))

np.int64(224)

In [ ]:
# 1) Load wavelengths from any one EnMAP JSON sidecar (make sure order matches bands)
with open("path/to/one_scene/…SPECTRAL_IMAGE_COG.json") as f:
    meta = json.load(f)
# adjust the path below to wherever wavelengths live in your JSON:
wl_nm = np.array(meta["sensor"]["spectral"]["wavelength_center_nm"], dtype=float)

# 2) Configure
cfg = SpectralCleanConfig(
    wl_nm=wl_nm,
    keep_ranges_nm=((430,1330),(1460,1780),(1960,2450)),
    hampel_window=5, hampel_nsig=4.0,
    sg_window=7, sg_poly=2,
    scene_robust_scale=True,
    groupby_col="scene_id",
    min_coverage_keep=0.7,                 # tighten as you like
    qa_drop_cols=["cloud", "shadow", "snow", "saturated"],  # if present
    stats_path=Path("artifacts/spectral_clean_stats.json"),
)

cleaner = SpectralCleaner(cfg)

# 3) Training/assembly split:
#    - Fit + transform on your training build
df_train_clean = cleaner.fit_transform(df_train)

#    - Save stats (already saved if cfg.stats_path is set)
cleaner.save_stats()

# 4) Apply to eval/test later (reuses per-scene stats; unseen scenes get fallback stats)
cleaner2 = SpectralCleaner(cfg)
cleaner2.load_stats()
df_test_clean = cleaner2.transform(df_test)
